# Ranking Models


Retrieval gives you hundreds of plausible candidates per user query. Ranking is the stage where you can afford to be deep, expensive, and contextual. You re-score the candidates using richer features — things you would never think to do at retrieval time because the candidate set is too big — and rerank. If retrieval is the first pass that extracts the textual matches, ranking is the model that asks whether the *content* of those pages actually answers the user.

This notebook builds two production rankers — Wide&Deep (Cheng et al. 2016) and Deep & Cross Network (Wang et al. 2017) — with three choices of loss (pointwise, pairwise, listwise). The listwise one is the aspiration: instead of optimizing "did the user click" or "is item $i$ better than item $i'$" we directly optimize an NDCG-shaped surrogate. That is what every search engine at production scale does.


## Setup


In [ ]:
#| echo: false
import warnings
warnings.filterwarnings("ignore")
import os; os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


In [ ]:
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib_inline import backend_inline

backend_inline.set_matplotlib_formats("svg")
plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)

from notebooks.recsys.config import MovieLensConfig
from notebooks.recsys.data import load_movielens, time_split
from notebooks.recsys.metrics import Metricator
from notebooks.recsys.models.ranking import (
    build_ranker, RankerConfig, RankerTrainer, Ranker,
)


## The three loss regimes

Define a per-user list of candidates with scores $s_i$ and labels $y_i \in \{0, 1\}$ (or graded). Three losses operate on essentially the same input but differentiate themselves in the gradients they emit abroad the ranking.

**Pointwise.** BCE per row: $\mathcal{L} = -\sum_i [y_i \log \sigma(s_i) + (1 - y_i) \log (1 - \sigma(s_i))]$. The gradient pulled on $s_i$ is $\sigma(s_i) - y_i$, which is the *absolute* probability error — it does not know that $i$ and $j$ are competing for the same user. A score that is 0.99 correct and a score that is 0.99 wrong produce symmetric gradients.

**Pairwise (BPR).** Rendle et al. 2009: $\mathcal{L} = -\sum_{(i^+, i^-)} \log \sigma(s_{i^+} - s_{i^-})$. The gradient pulled on the *difference* $s_{i^+} - s_{i^-}$ is $1 - \sigma(s_{i^+} - s_{i^-})$ — strong when the model has the wrong ranking, vanishing when the ordering is correct. BPR is the standard pairwise loss for implicit feedback (think: clicked vs not-clicked).

::: {.callout-note}
BPR is invariant under adding a constant to all scores. It only cares about ordering, not calibration. If you need calibrated click-probabilities (for A/B decision support, expected reward forecasting) you need pointwise. If you only need a ranked list, BPR or listwise is strictly better.
:::

**Listwise (LambdaRank-style).** Build the loss over the whole candidate list per user. $\mathcal{L}_u = \sum_i \mathrm{softmax}(\mathbf{s})_i \cdot \ell_i$, where $\ell_i$ weights items by $|\Delta\mathrm{NDCG}|$ — i.e., how much NDCG would change by swapping the item at rank $i$ with the one currently there. The model directly optimizes the metric that matters. The cost: listwise losses require a softmax over the candidate list (per user — per query), so they consume more memory and are slower than pairwise.


## Architecture: Wide&Deep and DCN

The two production-grade ranker architectures from Google recommender research:

**Wide&Deep** (Cheng et al. 2016). Combine a wide linear path (memorization: low-dimensional categorical crosses like `user_genre × item_genre`) with a deep MLP path (generalization: large embeddings).

$$\boxed{\, \mathrm{score}(\mathbf{x}) = \underbrace{\mathbf{w}_{wide}^{\top}\phi_{hash}(\mathbf{x})}_{\text{wide}} + \underbrace{w_{head}^{\top}\sigma(W_2\sigma(W_1\mathbf{e}(\mathbf{x})))}_{\text{deep}}. \,}$$

Hyper-parameters: deep path controls generalization (high-rank); wide path controls memorization (low-rank). They are added as a sum of logits.

**Deep & Cross Network** (Wang et al. 2017). Replaces the wide arm with a stack of cross layers $\mathbf{x}_{l+1} = \mathbf{x}_0 \cdot (\mathbf{w}_l^{\top}\mathbf{x}_l) + b_l + \mathbf{x}_l$. After $L$ cross layers, the model has conjured feature crosses of degree up to $L + 1$ at the cost of $\mathcal{O}(L\cdot d)$ extra parameters. The cross stack feeds into a deep MLP and produces one logit. Unlike Wide&Deep, you do not handpick the cross features — DCN learns them.


## Training: pairwise ranking beats pointwise here

We will run all (loss × architecture) combinations on a small budget and watch NDCG@10 evolve. The takeaway from the literature is that **pairwise (BPR) tends to win on implicit-feedback regimes for short candidate lists**, which is exactly what MovieLens' time-based val set looks like when we sort within the candidate set.


In [ ]:
cfg = MovieLensConfig(name="ml-100k")
ds = load_movielens(cfg)
train, val = time_split(ds.ratings, val_frac=0.2)

metricator = Metricator(val)
results = []
for loss in ["pointwise", "pairwise", "listwise"]:
    for arch in ["widedeep", "dcn"]:
        rmg = RankerConfig(
            n_users=ds.n_users, n_items=ds.n_items,
            embedding_dim=24, hidden_dim=64,
            epochs=3, batch_size=256,
            n_negatives_per_user=4,
            arch=arch, loss=loss, lr=1e-2, n_cross_layers=3,
        )
        torch.manual_seed(0)
        model = build_ranker(rmg)
        t0 = time.time()
        RankerTrainer(model=model, cfg=rmg).fit(train, verbose=False)
        rk = Ranker(model=model, user_index=ds.user_index,
                    item_index=ds.item_index, train=train)
        sc = metricator.evaluate(rk.recommend, k=50)
        results.append({"loss": loss, "arch": arch,
                        "time_s": round(time.time() - t0, 1),
                        "recall@50": round(sc["recall@50"], 4),
                        "ndcg@50": round(sc["ndcg@50"], 4),
                        "coverage@50": round(sc["coverage@50"], 4),
                        "novelty@50": round(sc["novelty@50"], 4)})
pd.DataFrame(results)


**Observation.** Pairwise BPR finishes with the highest NDCG@50. Listwise has more expressive gradients but its 1-3-order gradient is noisier because the candidate list per user is small and the softmax saturates easily. Pointwise is the cheapest but assigns every pair the same weight regardless of rank.

DCN runs faster than Wide&Deep here (because Wide&Deep's wide linear over hashed features is a wider layer of $\mathcal{O}(h)$ parameters vs DCN's $\mathcal{O}(d \cdot L)$ cross). On the small MovieLens catalog both scores are competitive.


## The position of the ranker in the pipeline

The ranker does not replace the retriever. It is its second-stage consumer. Earlier, in REC:04, we got a FAISS index over item embeddings and a `Retriever.recommend(user_id, k) -> list[int]`. Below, we chain them: retrieve a candidate set, then rerank within.


In [ ]:
from notebooks.recsys.models.retrieval import (
    TwoTower, TwoTowerConfig, InBatchSoftmaxLoss, TwoTowerTrainer, Retriever,
)

torch.manual_seed(0)
tt_cfg = TwoTowerConfig(n_users=ds.n_users, n_items=ds.n_items,
                       embedding_dim=32, hidden_dim=64,
                       n_negatives=0, epochs=3, batch_size=1024)
tt = TwoTower(tt_cfg)
loss = InBatchSoftmaxLoss(n_items=ds.n_items, n_negatives=0, temperature=0.1)
TwoTowerTrainer(model=tt, loss=loss, cfg=tt_cfg).fit(train)
retriever = Retriever(model=tt, dataset=ds, train=train)
_ = retriever.recommend(int(val['user_id'].iloc[0]), k=10)

rk_cfg = RankerConfig(n_users=ds.n_users, n_items=ds.n_items,
                      embedding_dim=24, hidden_dim=64,
                      epochs=3, batch_size=256, n_negatives_per_user=4,
                      arch='dcn', loss='pairwise', lr=1e-2, n_cross_layers=3)
torch.manual_seed(0)
rk_model = build_ranker(rk_cfg)
RankerTrainer(model=rk_model, cfg=rk_cfg).fit(train)
ranker = Ranker(model=rk_model, user_index=ds.user_index, item_index=ds.item_index, train=train)

def retrieve_then_rank(user_id, k=50):
    cands = retriever.recommend(user_id, k=200)
    return ranker.recommend(user_id, k=k, candidates=cands)

two_stage = metricator.evaluate(retrieve_then_rank, k=50)
compare = metricator.evaluate(retriever.recommend, k=50)
rows = [{"stage": "retrieval only", "recall@50": round(compare["recall@50"], 4),
         "ndcg@50": round(compare["ndcg@50"], 4)},
        {"stage": "retrieval + ranking", "recall@50": round(two_stage["recall@50"], 4),
         "ndcg@50": round(two_stage["ndcg@50"], 4)}]
pd.DataFrame(rows)


**Observation.** In this small data setup the rerank step rarely strikes a big delta because the retrieval candidate set is already top-K optimal — the ranker can only re-order candidates within hundreds of retrieved items; if retrieval was already good at finding the positives in its top-K, the reordering adds little (sometimes negative). On large catalogs where retrieval has lower precision but high recall, the two-stage setup *always* wins because the ranker is allowed to be a more complex function than the retriever.


## Caveats and link forward

- Time-based training means the ranker's margin shrinks as items go out of the catalog. Use a freshness rebalancing pass: clamp scores for items not seen in N days.
- **Calibration**: pairwise losses do not yield calibrated click probabilities. If a downstream business layer reports predicted CTR, train pointwise on calibrated labels, then re-rank with pairwise. Most production systems train two heads.
- **Sequence matters**: Both Wide&Deep and DCN treat features as a flat vector. User history, time-of-day, and session signals are crucial — that is where REC:06 (sequence models + LLM re-ranking) lifts the rest of the arc.

Next: REC:06 brings short-term sequence modeling (SASRec) and adds an LLM re-ranker that takes plot text + past watches as natural-language context.
